In [16]:
# Cell 1 — Imports

from pathlib import Path
import yaml
import pandas as pd
import xarray as xr

In [17]:


PROJECT_ROOT = Path.cwd().parent
RUN_CONFIG = PROJECT_ROOT / "config.run.yaml"
print("Project root:", PROJECT_ROOT)
print("Run config:", RUN_CONFIG)
print("Exists:", RUN_CONFIG.exists())

Project root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape
Run config: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/config.run.yaml
Exists: True


In [18]:
# Cell 3 — Load run configuration

with open(RUN_CONFIG, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

# cfg

In [19]:
# Cell 4 — Run scope summary

print("Region:", cfg["region"]["name"])
print("CRS:", cfg["region"]["crs"])
print("BBox:", cfg["region"]["bbox"])
print("Buffer (km):", cfg["region"]["buffer_km"])
print("Grid:", cfg["grid"]["system"], cfg["grid"]["resolution"])
print("Time:", cfg["time"]["start"], "->", cfg["time"]["end"])

Region: falkland_islands
CRS: EPSG:4326
BBox: {'xmin': -64, 'ymin': -57, 'xmax': -51, 'ymax': -47}
Buffer (km): 50
Grid: H3 6
Time: 2014-01-01 -> 2023-12-31


In [20]:
# Cell 6 — Discover raw datasets from data/raw

raw_root = PROJECT_ROOT / cfg["paths"]["raw"]

dataset_dirs = sorted([p for p in raw_root.iterdir() if p.is_dir()])

print("Raw root:", raw_root)
print("Datasets found:", len(dataset_dirs))

for d in dataset_dirs:
    print(" ", d.name)

Raw root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw
Datasets found: 4
  chl
  ssh
  sst
  wind


In [21]:
# Cell 7 — Discover raw datasets and files

raw_root = PROJECT_ROOT / cfg["paths"]["raw"]

dataset_dirs = sorted([p for p in raw_root.iterdir() if p.is_dir()])

datasets = {}

for dataset_dir in dataset_dirs:
    files = sorted(
        [
            p for p in dataset_dir.rglob("*")
            if p.is_file() and not p.name.startswith(".")
        ]
    )
    datasets[dataset_dir.name] = {
        "directory": dataset_dir,
        "files": files,
    }

print("Raw root:", raw_root)
print("Datasets found:", len(datasets))

for name, meta in datasets.items():
    print(f"  {name}: {len(meta['files'])} files")

Raw root: /Users/pb/Work/OSU/CapstoneProject/bycatch-riskscape/data/raw
Datasets found: 4
  chl: 1 files
  ssh: 1 files
  sst: 3652 files
  wind: 120 files


In [23]:
# Cell 8 — Inspect all datasets (clean summary)

import tempfile
import zipfile


def summarize_xarray_dataset(path):
    """Return a compact summary of an xarray-readable dataset."""
    with xr.open_dataset(path) as ds:
        time_coverage = None

        for name in ds.coords:
            coord = ds.coords[name]
            if str(coord.dtype).startswith("datetime64"):
                time_coverage = (
                    str(coord.min().values),
                    str(coord.max().values),
                )
                break

        return {
            "dimensions": dict(ds.sizes),
            "variables": list(ds.data_vars),
            "time_coverage": time_coverage,
        }


def inspect_dataset_file(path):
    """Inspect a dataset file or archive and return a compact summary."""
    suffix = path.suffix.lower()

    info = {
        "sample_file": path.name,
        "container": suffix.lstrip("."),
        "dimensions": None,
        "variables": None,
        "time_coverage": None,
        "note": None,
    }

    if suffix in {".nc", ".nc4", ".cdf"}:
        summary = summarize_xarray_dataset(path)
        info.update(summary)
        return info

    if suffix == ".zip":
        with zipfile.ZipFile(path, "r") as z:
            members = [m for m in z.namelist() if not Path(m).name.startswith(".")]
            nc_members = [
                m for m in members
                if Path(m).suffix.lower() in {".nc", ".nc4", ".cdf"}
            ]

            if not nc_members:
                info["note"] = "ZIP contains no NetCDF file"
                return info

            member = nc_members[0]

            with tempfile.TemporaryDirectory() as tmpdir:
                extracted = Path(z.extract(member, path=tmpdir))
                summary = summarize_xarray_dataset(extracted)

            info.update(summary)
            info["note"] = f"ZIP archive with {len(members)} files"
            return info

    info["note"] = f"Unsupported file type: {suffix}"
    return info


for name, meta in datasets.items():
    files = [p for p in meta["files"] if not p.name.startswith(".")]

    print("=" * 80)
    print(f"Dataset: {name}")
    print(f"File count: {len(files)}")

    if not files:
        print("Note: No files found")
        continue

    try:
        info = inspect_dataset_file(files[0])

        print(f"Sample file: {info['sample_file']}")
        print(f"Container: {info['container']}")

        if info["dimensions"] is not None:
            print(f"Dimensions: {info['dimensions']}")

        if info["variables"] is not None:
            print(f"Variables: {info['variables']}")

        if info["time_coverage"] is not None:
            start, end = info["time_coverage"]
            print(f"Time coverage: {start} -> {end}")

        if info["note"]:
            print(f"Note: {info['note']}")

    except Exception as exc:
        print(f"Note: ERROR: {exc}")

Dataset: chl
File count: 1
Sample file: cmems_obs-oc_glo_bgc-plankton_my_l4-gapfree-multi-4km_P1D_CHL_64.73W-50.27W_57.44S-46.56S_2014-01-01-2023-12-31.nc
Container: nc
Dimensions: {'time': 3652, 'latitude': 262, 'longitude': 348}
Variables: ['CHL']
Time coverage: 2014-01-01T00:00:00.000000000 -> 2023-12-31T00:00:00.000000000
Dataset: ssh
File count: 1
Sample file: cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt_64.69W-50.31W_57.44S-46.56S_2014-01-01-2023-12-31.nc
Container: nc
Dimensions: {'time': 3652, 'latitude': 88, 'longitude': 116}
Variables: ['adt']
Time coverage: 2014-01-01T00:00:00.000000000 -> 2023-12-31T00:00:00.000000000
Dataset: sst
File count: 3652
Sample file: sst_20140101.nc
Container: nc
Dimensions: {'time': 1, 'lat': 1091, 'lon': 1447}
Variables: ['analysed_sst']
Time coverage: 2014-01-01T09:00:00.000000000 -> 2014-01-01T09:00:00.000000000
Dataset: wind
File count: 120
Sample file: derived-era5-single-levels-daily-statistics_2014_01.zip
Container: zip
Dim